## Data Processing Notebook

In [ ]:
## Imports

import os

from sqlalchemy import create_engine, text
import pandas as pd

In [2]:
rurality_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_attainment_by_rurality.csv"))
chars_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_attainment_by_characteristics.csv"))
retention_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_retention_by_region.csv"))
results_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_results_by_subject.csv"))
stem_path = os.path.abspath(os.path.join("NBO1_Data_Collection.ipynb/"+"../.."+"/data/raw/EES_stem_by_sex.csv"))

In [3]:
rurality_df = pd.read_csv(rurality_path)
chars_df = pd.read_csv(chars_path)
retention_df = pd.read_csv(retention_path)
results_df = pd.read_csv(results_path)
stem_df = pd.read_csv(stem_path)

In [ ]:
## Define engine for SQLite database

engine = create_engine('sqlite:///../data/database.db')

In [ ]:
## Define schemas for each table

## Rurality table
ees_attainment_by_rurality_schema = """
CREATE TABLE rurality (
    time_period VARCHAR(10) NOT NULL,
    region VARCHAR(25) NOT NULL,
    rurality VARCHAR(40) NOT NULL,
    number_of_students_entered INTEGER,
    number_of_a_level_students INTEGER,
    number_of_students_completed INTEGER,
    perc_entering_a_levels DECIMAL(5,2),
    perc_achieving_atleast_two_a_levels DECIMAL(5,2),
    perc_achieving_3_astar_to_a_grades DECIMAL(5,2),
    average_a_level_grade VARCHAR(2),
    PRIMARY KEY (time_period, region, rurality)
)
"""

## Characteristics table
ees_attainment_by_characteristics_schema = """
CREATE TABLE  characteristics (
    time_period VARCHAR(10) NOT NULL,
    region VARCHAR(25) NOT NULL,
    characteristic_type VARCHAR(15) NOT NULL,
    characteristic_value VARCHAR(30) NOT NULL,
    number_of_students_entered INTEGER,
    number_of_a_level_students INTEGER,
    number_of_students_completed INTEGER,
    perc_entering_a_levels DECIMAL(5,2),
    perc_achieving_atleast_two_a_levels DECIMAL(5,2),
    perc_achieving_3_astar_to_a_grades DECIMAL(5,2),
    average_a_level_grade VARCHAR(5),
    PRIMARY KEY (time_period, region, characteristic_type, characteristic_value)
)
"""

## Retention table
ees_retention_by_region_schema = """
CREATE TABLE retention (
    time_period VARCHAR(10) NOT NULL,
    region VARCHAR(25) NOT NULL,
    cohort VARCHAR(20) NOT NULL,
    student_count_year_1 INTEGER,
    student_count_year_2 INTEGER,
    retained INTEGER,
    retained_and_assessed INTEGER,
    returned_and_retained INTEGER,
    perc_retained DECIMAL(5,2),
    perc_retained_and_assessed DECIMAL(5,2),
    perc_returned_and_retained DECIMAL(5,2),
    PRIMARY KEY (time_period, region, cohort)
)
"""

## Results table
ees_results_by_subject_schema = """
CREATE TABLE results (
    time_period VARCHAR(10) NOT NULL,
    region VARCHAR(25) NOT NULL,
    qualification VARCHAR(10) NOT NULL,
    subject_area VARCHAR(100) NOT NULL,
    subject_name VARCHAR(100) NOT NULL,
    number_of_students_entered INTEGER,
    astar_grade_achieved INTEGER,
    a_grade_achieved INTEGER,
    b_grade_achieved INTEGER,
    c_grade_achieved INTEGER,
    d_grade_achieved INTEGER,
    e_grade_achieved INTEGER,
    u_grade_achieved INTEGER,
    astar_to_a_grade_achieved INTEGER,
    astar_to_b_grade_achieved INTEGER,
    astar_to_c_grade_achieved INTEGER,
    astar_to_d_grade_achieved INTEGER,
    astar_to_e_grade_achieved INTEGER,
    PRIMARY KEY (time_period, region, qualification, subject_area, subject_name)
)
"""

## STEM table
ees_stem_by_sex_schema = """
CREATE TABLE stem (
    time_period VARCHAR(10) NOT NULL,
    region VARCHAR(25) NOT NULL,
    sex VARCHAR(10) NOT NULL,
    sub_name_comb VARCHAR(200) NOT NULL,
    num_maths_science INTEGER,
    number_of_students_entered INTEGER,
    perc_entered_comb DECIMAL(5,2),
    PRIMARY KEY (time_period, region, sex, sub_name_comb)
)
"""

with engine.connect() as conn:
    conn.execute(text(ees_attainment_by_rurality_schema))
    conn.execute(text(ees_attainment_by_characteristics_schema))
    conn.execute(text(ees_retention_by_region_schema))
    conn.execute(text(ees_results_by_subject_schema))
    conn.execute(text(ees_stem_by_sex_schema))

    conn.commit()


In [ ]:
## Mapping dictionaries for each table, based on your schemas and CSV data

## Rurality table
rurality_column_map = {
    "time_period": "time_period",
    "region_name": "region",
    "rurality_name": "rurality",
    "number_of_students_entered": "number_of_students_entered",
    "number_of_students_alev": "number_of_a_level_students",
    "number_of_students_potential": "number_of_students_completed",
    "pc_entering_alev": "perc_entering_a_levels",
    "pc_achieving_atleast_two_alev": "perc_achieving_atleast_two_a_levels",
    "pc_achieving_3_astar_to_a_alev": "perc_achieving_3_astar_to_a_grades",
    "aps_per_entry_grade_alev": "average_a_level_grade"
}

## Characteristics table
chars_column_map = {
    "time_period": "time_period",
    "region_name": "region",
    "characteristic_type": "characteristic_type",
    "characteristic_value": "characteristic_value",
    "number_of_students_entered": "number_of_students_entered",
    "number_of_students_alev": "number_of_a_level_students",
    "number_of_students_potential": "number_of_students_completed",
    "pc_entering_alev": "perc_entering_a_levels",
    "pc_achieving_atleast_two_alev": "perc_achieving_atleast_two_a_levels",
    "pc_achieving_3_astar_to_a_alev": "perc_achieving_3_astar_to_a_grades",
    "aps_per_entry_grade_alev": "average_a_level_grade"
}

## Retention table
retention_column_map = {
    "time_period": "time_period",
    "region_name": "region",
    "exam_cohort": "cohort",
    "student_count_year_1": "student_count_year_1",
    "student_count_year_2": "student_count_year_2",
    "retained": "retained",
    "retained_and_assessed": "retained_and_assessed",
    "returned_and_retained": "returned_and_retained",
    "perc_retained": "perc_retained",
    "perc_retained_and_assessed": "perc_retained_and_assessed",
    "perc_returned_and_retained": "perc_returned_and_retained"
}

## Results table
results_column_map = {
    "time_period": "time_period",
    "region_name": "region",
    "qualification": "qualification",
    "subject_area": "subject_area",
    "subject_name": "subject_name",
    "entry_count": "number_of_students_entered",
    "astar_grade_achieved": "astar_grade_achieved",
    "a_grade_achieved": "a_grade_achieved",
    "b_grade_achieved": "b_grade_achieved",
    "c_grade_achieved": "c_grade_achieved",
    "d_grade_achieved": "d_grade_achieved",
    "e_grade_achieved": "e_grade_achieved",
    "u_grade_achieved": "u_grade_achieved",
    "astar_a_grade_achieved": "astar_to_a_grade_achieved",
    "astar_b_grade_achieved": "astar_to_b_grade_achieved",
    "astar_c_grade_achieved": "astar_to_c_grade_achieved",
    "astar_d_grade_achieved": "astar_to_d_grade_achieved",
    "astar_e_grade_achieved": "astar_to_e_grade_achieved"
}

## STEM table
stem_column_map = {
    "time_period": "time_period",
    "region_name": "region",
    "characteristic_sex": "sex",
    "sub_name_comb": "sub_name_comb",
    "num_maths_science": "num_maths_science",
    "num_entered_comb_and_no_other_matsci": "number_of_students_entered",
    "perc_entered_comb_and_no_other_matsci": "perc_entered_comb",
}

rurality_df.rename(columns=rurality_column_map, inplace=True)
chars_df.rename(columns=chars_column_map, inplace=True)
retention_df.rename(columns=retention_column_map, inplace=True)
results_df.rename(columns=results_column_map, inplace=True)
stem_df.rename(columns=stem_column_map, inplace=True)

In [ ]:
## Insert data into the database tables

rurality_df.to_sql('rurality', engine, if_exists='append', index=False)
chars_df.to_sql('characteristics', engine, if_exists='append', index=False)
retention_df.to_sql('retention', engine, if_exists='append', index=False)
results_df.to_sql('results', engine, if_exists='append', index=False)
stem_df.to_sql('stem', engine, if_exists='append', index=False)

9450

In [8]:
conn.close()
engine.dispose()